In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
# first-time use execution
!pip install biopython

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
!pip install matplotlib ipywidgets --target=$nb_path

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

In [ ]:
# INSTALL pymesh!!
# upload the .whl file for installation
# https://github.com/nuvolos-cloud/PyMesh/releases
from google.colab import files
uploaded = files.upload()
## install pymesh to calculate the curvature of the protein surface
!pip install pymesh2-0.3.1-cp310-cp310-linux_x86_64.whl --target=$nb_path  --upgrade

In [ ]:
import pymesh


In [ ]:
#@title TEST generate mesh for all the atoms of pdb
import pandas as pd
import numpy as np
import pymesh
from Bio.PDB import PDBParser, PDBIO

def calculate_curvature(pdb_file_path, n_neighbors):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_file_path)
    model = structure[0]
    chain = model['C']

    atoms = [atom for atom in chain.get_atoms()]
    coords = np.array([atom.coord for atom in atoms])
    curvatures = [None] * len(atoms)

    for idx, atom in enumerate(atoms):
        atom_coords = np.array(coords[idx])
        dist_coords = coords - atom_coords

        # Calculate distances and get indices of nearest neighbors
        dists = np.linalg.norm(dist_coords, axis=1)
        nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]

        # Create vertices and faces
        vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

        faces = [[0, i, i + 1] for i in range(1, n_neighbors)] + [[0, n_neighbors, 1]]


        faces = np.array(faces)

        # Create mesh
        mesh = pymesh.form_mesh(vertices, faces)

        mesh.add_attribute("vertex_mean_curvature")
        H = mesh.get_attribute("vertex_mean_curvature")
        mean_H = H[0]

        mesh.add_attribute("vertex_gaussian_curvature")
        K = mesh.get_attribute("vertex_gaussian_curvature")
        K = K[0]

        elem = np.square(mean_H) - K
        elem = np.array(elem)

        k1 = mean_H + np.sqrt(elem)
        k2 = mean_H - np.sqrt(elem)

        si = (k1 + k2) / (k1 - k2)
        si = np.arctan(si) * (2 / np.pi)

        curvatures[idx] = si

    return curvatures

# Function to update B-factor with curvature values
def select_and_update(structure, curvatures):
    for model in structure:
        for chain in model:
            if chain.id == 'C':
                for i, residue in enumerate(chain):
                    for j, atom in enumerate(residue):
                        if i * len(residue) + j < len(curvatures):
                            curvature_value = curvatures[i * len(residue) + j]
                            if curvature_value is not None:
                                atom.set_bfactor(curvature_value)

# Initialize a PDB parser
parser = PDBParser(QUIET=True)

# Load the structure
pdb_file_path = '1a2y_B_A_C.pdb'
structure = parser.get_structure('1a2y_B_A_C', pdb_file_path)

# Calculate curvatures
n_neighbors_list = [6]
for n_neighbors in n_neighbors_list:
    curvatures = calculate_curvature(pdb_file_path, n_neighbors)

    # Create a new PDBIO object for saving the modified structure
    io = PDBIO()

    # Save the updated chain C structure
    output_path = f'1a2y_B_A_C_chain_C_with_curvature_{n_neighbors}.pdb'
    io.set_structure(structure)
    select_and_update(structure, curvatures)
    io.save(output_path)


In [ ]:
from numpy.core.fromnumeric import sort
import pandas as pd
import numpy as np
import pymesh
from Bio.PDB import PDBParser, PDBIO
pdb_file_path = '1a2y_B_A_C.pdb'
parser = PDBParser(QUIET=True)
structure = parser.get_structure('protein', pdb_file_path)
model = structure[0]
chain = model['C']
atoms = [atom for atom in chain.get_atoms()]
coords = np.array([atom.coord for atom in atoms])
curvatures = [None] * len(atoms)
n_neighbors = 5
for idx, atom in enumerate(atoms[67:68]):
  atom_coords = np.array(coords[idx])

  dist_coords = coords - atom_coords
  # Calculate distances and get indices of nearest neighbors
  dists = np.linalg.norm(dist_coords, axis=1)
  nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]

  # Create vertices and faces
  vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]
  faces = [[0, i, i + 1] for i in range(1, n_neighbors)] + [[0, n_neighbors, 1]]


  faces = np.array(faces)

  # Create mesh
  mesh = pymesh.form_mesh(vertices, faces)

  mesh.add_attribute("vertex_mean_curvature")
  H = mesh.get_attribute("vertex_mean_curvature")
  mean_H = H[0]

  mesh.add_attribute("vertex_gaussian_curvature")
  K = mesh.get_attribute("vertex_gaussian_curvature")
  K = K[0]

  if K < 0:
    K = 1e-8


  k1 = mean_H + np.sqrt(K)
  k2 = mean_H - np.sqrt(K)
  si = (k1 + k2) / (k1 - k2)
  si = np.arctan(si) * (2 / np.pi)

  curvatures[idx] = si
  print(vertices)
  print(si)


In [ ]:
#@title TEST Method_2 pymol generate mesh object and then
from pymol import cmd

In [ ]:
from pymol import cmd



pdb_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/1a2y_B_A_C.pdb'
obj_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/model_chain_c.obj'  # Replace with your desired output path

# Load the PDB file into an object named 'protein'
cmd.load(pdb_file_path, 'protein')

# Create a selection for only the CA atoms of chain C
cmd.select('chain_c', 'chain C and polymer')

# Show the surface for the selected CA atoms
cmd.show('surface', 'chain_c')

# Save the view as a .png file
cmd.save(obj_file_path, 'chain C', format='obj')




In [ ]:
import pymesh
import numpy as np

# Load mesh
mesh = pymesh.load_mesh(obj_file_path)

# Enable attributes for curvature calculations
mesh.add_attribute("vertex_mean_curvature")
mesh.add_attribute("vertex_gaussian_curvature")

# Retrieve the curvature attributes
H = mesh.get_attribute("vertex_mean_curvature")
K = mesh.get_attribute("vertex_gaussian_curvature")

# Calculate mean curvature (H) and Gaussian curvature (K)
mean_H = np.mean(H)
mean_K = np.mean(K)

# Calculate principal curvatures k1 and k2
elem = np.square(mean_H) - mean_K
elem = np.array(elem)
elem[elem < 0] = 1e-8  # Handle negative values
k1 = mean_H + np.sqrt(elem)
k2 = mean_H - np.sqrt(elem)

# Compute the shape index for each vertex
si = (2/np.pi) * np.arctan((k1 + k2) / (k1 - k2))

# Initialize curvatures list
curvatures = np.zeros(mesh.num_vertices)

# Since we used the mean curvature and mean Gaussian curvature, the shape index would be the same for all vertices.
# Therefore, we populate the 'curvatures' list with the same shape index value.
curvatures.fill(si)

print(f"Curvatures: {curvatures}")


In [ ]:
len(H)

In [ ]:
# visualize the obj file
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interactive

def read_obj(filename):
    vertices = []
    faces = []
    with open(filename, 'r') as f:
        for line in f:
            parts = line.strip().split(" ")
            if parts[0] == "v":
                vertices.append([float(v) for v in parts[1:]])
            elif parts[0] == "f":
                faces.append([int(f.split('/')[0]) for f in parts[1:]])
    return np.array(vertices), np.array(faces)

# Replace with your .obj file path
obj_file_path = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_curvature_testing/model_chain_c.obj'
vertices, faces = read_obj(obj_file_path)

def plot_figure(elev, azim):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.view_init(elev=elev, azim=azim)

    for face in faces:
        for i in range(len(face)):
            ax.plot([vertices[face[i] - 1][0], vertices[face[i - 1] - 1][0]],
                    [vertices[face[i] - 1][1], vertices[face[i - 1] - 1][1]],
                    [vertices[face[i] - 1][2], vertices[face[i - 1] - 1][2]], 'b-')
    plt.show()

interactive_plot = interactive(plot_figure, elev=widgets.IntSlider(min=0, max=180, step=10, value=30), azim=widgets.IntSlider(min=0, max=360, step=10, value=30))
output = interactive_plot.children[-1]
interactive_plot
